# COE 2022 "GPRA Friendly" Fixed (1 run)

If you want to run this notebook, keep in mind that the library path from the FLORIS config file has to be changed accordingly to your new path

## Imports and environment set up

In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt

import numpy as np

from whale import Project
from whale.utilities import load_yaml

pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

%load_ext autoreload
%autoreload 2

# Function to Extract Results

In [2]:
def extract_results(project_name, config):  # Run the project and clean up the logging
    #project_name = "UMaine_catenary"
    library_path = Path("library").resolve()
    
    metrics_configuration = {
        "# Turbines": {"metric": "n_turbines", "kwargs": {}},
        "Turbine Rating (MW)": {"metric": "turbine_rating", "kwargs": {}},
        "Project Capacity (MW)": {"metric": "capacity", "kwargs": {"units": "mw"}},
        "# OSS": {"metric": "n_substations", "kwargs": {}},
        "Total Export Cable Length (km)": {"metric": "export_system_total_cable_length", "kwargs": {}},
        "Total Array Cable Length (km)": {"metric": "array_system_total_cable_length", "kwargs": {}},
        "CapEx ($)": {"metric": "capex", "kwargs": {}},
        "CapEx per kW ($/kW)": {"metric": "capex", "kwargs": {"per_capacity": "kw"}},
        "OpEx ($)": {"metric": "opex", "kwargs": {}},
        "OpEx per kW ($/kW)": {"metric": "opex", "kwargs": {"per_capacity": "kw"}},
        "AEP (MWh)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "aep": True, "with_losses": True}
        },
        "AEP per kW (MWh/kW)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "per_capacity": "kw", "aep": True, "with_losses": True}
        },
        "Net Capacity Factor Without Unmodeled Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net"}
        },
        "Net Capacity Factor With All Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net", "with_losses": True}
        },
        "Gross Capacity Factor (%)": {"metric": "capacity_factor", "kwargs": {"which": "gross"}},
        "Energy Availability (%)": {"metric": "availability", "kwargs": {"which": "energy"}},
        "LCOE ($/MWh)": {"metric": "lcoe", "kwargs": {}},
        "IRR (%)": {"metric": "irr", "kwargs": {}},
        "NPV ($)": {"metric": "npv", "kwargs": {}},
    }

    metrics_order = [
        "# Turbines",
        "Turbine Rating (MW)",
        "Project Capacity (MW)",
        "# OSS",
        "Total Export Cable Length (km)",
        "Total Array Cable Length (km)",
        "FCR (%)",
        "Offtake Price ($/MWh)",
        "CapEx ($)",
        "CapEx per kW ($/kW)",
        "System CapEx for Export Cables ($)",
        "Installation CapEx for Export Cables ($)",
        "CapEx Without Export Cables ($)",
        "OpEx ($)",
        "OpEx per kW ($/kW)",
        "Annual OpEx per kW ($/kW)",
        "Energy Availability (%)",
        "Gross Capacity Factor (%)",
        "Net Capacity Factor Without Unmodeled Losses (%)",
        "Net Capacity Factor With All Losses (%)",
        "AEP (MWh)",
        "AEP per kW (MWh/kW)",
        "LCOE ($/MWh)",
        "IRR (%)",
        "NPV ($)",
    ]
    final_cols = ["CapEx ($)", "OpEx ($)", "Energy Production (GWh)", "Revenue ($)", "Cash Flow ($)"]
    
    #config = load_yaml(
        #library_path / "project/config",
        #f"{project_name.replace(' ', '_')}_base.yaml"
    #)

    #wombat_config = load_yaml(
        #library_path / "project/config",
        #f"{project_name.replace(' ', '_')}_base_operations.yaml"
    #)
    
    
    #wombat_config['random_generator'] = np.random.default_rng(seed=34)
    
    #print(wombat_config)
    #config["wombat_config"] = wombat_config
    
    project = Project(
        # Basic Model Configurations
        library_path=library_path,
        #weather_profile=library_path / "weather" / "ocean_wind_1_39.0_-74.0_1959_2023.csv",
        connect_floris_to_layout=True,
        connect_orbit_array_design=True,
        **config,
    )

    
    project.run(
        which_floris="wind_rose",
        full_wind_rose=False,
        floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
    )
    project.wombat.env.cleanup_log_files()  # Delete logging data from WOMBAT
    

    
    #fig, ax = project.plot_farm(return_fig=True)
    #print("Monopile cost and transition piece costs")
    #print(project.orbit.phases['MonopileDesign'].detailed_output)

    df_times = project.wombat.metrics.process_times()

    display(df_times)

    scheduled = project.wombat.metrics.task_completion_rate(which="scheduled", frequency="project").values[0][0]
    unscheduled = project.wombat.metrics.task_completion_rate(which="unscheduled", frequency="project").values[0][0]
    combined = project.wombat.metrics.task_completion_rate(which="both", frequency="project").values[0][0]
    print(f"  Scheduled Task Completion Rate: {scheduled:.2%}")
    print(f"Unscheduled Task Completion Rate: {unscheduled:.2%}")
    print(f"    Overall Task Completion Rate: {combined:.2%}")
 
    df_capex  = pd.DataFrame(project.orbit.capex_breakdown.items(), columns=['CapEx Component', 'Value ($)'])

    df_capex["Value ($/kW)"] = df_capex["Value ($)"]/(project.capacity("mw")*1000)

    #display(df_capex)

    
    df_capex.to_csv('library/results/' + project_name + '_CapEx_Breakdown.csv')
    
    df_opex = project.wombat.metrics.opex(frequency="annual", by_category = True)
    
    #display(df_opex)
    
    df_opex.to_csv('library/results/' + project_name + '_OpEx_Breakdown.csv')
    
    list_of_dict = project.orbit.actions
    df = pd.DataFrame(list_of_dict)
    df.to_csv('library/results/' + project_name + '_Install_Sequence.csv')
    #display(df)
    
    # Gather the high-level results
    report_df = project.generate_report(metrics_configuration, project_name).T
    export_system = project.orbit.system_costs["ExportCableInstallation"]
    export_installation = project.orbit.installation_costs["ExportCableInstallation"]
    capex_sans_export = project.orbit.total_capex - export_system - export_installation
    additional_reporting = pd.DataFrame(
        [
            ["FCR (%)", project.fixed_charge_rate],
            ["Offtake Price ($/MWh)", project.offtake_price],
            ["System CapEx for Export Cables ($)", export_system],
            ["Installation CapEx for Export Cables ($)", export_installation],
            ["CapEx Without Export Cables ($)", capex_sans_export],
            ["Annual OpEx per kW ($/kW)", report_df.loc["OpEx per kW ($/kW)", project_name] / project.operations_years],
        ],
        columns=["Project"] + report_df.columns.tolist(),
    ).set_index("Project")
    report_df = pd.concat((report_df, additional_reporting), axis=0).loc[metrics_order]

    # Gather the detailed results
    monthly_results = project.cash_flow(breakdown=True).join(project.energy_production(frequency="month-year")).fillna(0)
    monthly_results = monthly_results.assign(
        CapEx_Installation=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("Installation")]].sum(axis=1),
        CapEx_System=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("System")]].sum(axis=1),
    )

    # monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_detailed_results.csv")
    monthly_results["CapEx ($)"] = monthly_results[["CapEx_Installation", "CapEx_Soft", "CapEx_Project", "CapEx_System", "CapEx_Turbine"]].sum(axis=1)
    monthly_results = monthly_results.rename(columns={"OpEx": "OpEx ($)","Revenue": "Revenue ($)", "cash_flow": "Cash Flow ($)"})[final_cols]

    if "ExportSystemDesign" in project.orbit._phases:
        export = "ExportSystemDesign"
    else:
        export = "ElectricalDesign"

    # Create the inputs data
    inputs = pd.DataFrame(
        [
            ["FCR", project.fixed_charge_rate],
            ["Discount rate (%)", project.discount_rate],
            ["# Turbines", project.n_turbines()],
            ["Turbine Rating (MW)", project.turbine_rating()],
            ["Project Capacity (MW)", project.capacity("mw")],
            ["# OSS", project.n_substations()],
            ["Substructure type", "??"],
            ["Row spacing (rotor diameters)", "not used for custom layouts"],
            ["Turbine spacing (rotor diameters)", "not used for custom layouts"],
            ["Depth (m)", project.orbit.config["site"]["depth"]],
            [
                "Mean wind speed (m/s)",
                project.weather.loc[
                    project.orbit_start_date: project.wombat.env.end_datetime,
                    "windspeed_100m"
                ].mean()
            ],
            ["Distance to landfall (km)", project.orbit.config["site"]["distance_to_landfall"]],
            ["Distance to port (km)", project.wombat.config.port_distance],
            ["Interconnection distance (km)", project.orbit._phases[export]._distance_to_interconnection],
            ["# of POIs", "??"],
            ["Export cable type", [*project.orbit._phases[export].cable_lengths_by_type]],
            ["Array cable type", [*project.orbit._phases["CustomArraySystemDesign"].cable_lengths_by_type]],
        ],
        columns=["Inputs", f"{project_name}"]
    ).set_index("Inputs")

    # Save the outputs
    report_df.index.name = "Metrics"
    wrong_indexes = ["Offtake Price ($/MWh)", "System CapEx for Export Cables ($)", "Installation CapEx for Export Cables ($)", "CapEx Without Export Cables ($)", "IRR (%)", "NPV ($)"]
    
    
    return report_df.drop(wrong_indexes,axis=0)

# Extract CapEx, OpEx, and AEP Outputs - Multiple Runs
Availability and OpEx varies depending on the run as it uses probabilistic functions, so we collect the data from x runs and calculate the average OpEx and AEP

In [3]:
def display_results_all_runs(list_projects):
    import pandas as pd
    library_path = Path("library").resolve()
    df_list_fixed_bottom = list()
    df_list_floating = list()
    df_final_list = list()
    num = 0
    for i in list_projects:
        config = load_yaml(
            library_path / "project/config",
            f"{i.replace(' ', '_')}_base.yaml"
        )

        wombat_config = load_yaml(
            library_path / "project/config",
            f"{i.replace(' ', '_')}_base_operations.yaml"
        )
    
    
        wombat_config['random_generator'] = np.random.default_rng(seed=34)
    
        print(wombat_config)
        config["wombat_config"] = wombat_config
        for j in range(0,1):
            
            df_2 = extract_results(i, config)
            
            if num == 0:
                df_list_floating = df_2
            else:
                df_list_floating = pd.concat([df_list_floating, df_2], axis=1)
                display(df_list_floating)
            
            num = num + 1        

            
    final_df = df_list_floating
                          
    return final_df

In [4]:
import time
t1 = time.perf_counter()

list_projects_15_MW = ["fixed_bottom_15_MW_-10%", "fixed_bottom_15_MW_-5%", "fixed_bottom_15_MW_baseline"]
summary_results = display_results_all_runs(list_projects_15_MW)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

{'name': 'fixed_bottom_15_MW_-10%_fixed_bottom', 'library': 'library', 'weather': 'boem_era5_41.0_-70.5.csv', 'service_equipment': ['ctv1.yaml', 'ctv2.yaml', 'ctv3.yaml', 'clv.yaml', 'jackup.yaml', 'dsv.yaml'], 'layout': 'fixed_bottom_layout_15_MW_-10%.csv', 'inflation_rate': 0, 'fixed_costs': 'fixed_bottom_fixed_costs.yaml', 'workday_start': 7, 'workday_end': 19, 'start_year': 2000, 'end_year': 2020, 'project_capacity': 600, 'port': 'port116.yaml', 'random_generator': Generator(PCG64) at 0x2759F722EA0}
ORBIT library intialized at 'C:\upsizing\WHaLE\examples\library'


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\upsizing\ORBIT\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,time_to_completion,process_time,downtime,time_to_start,N
category,,,,,
annual inspection,"472,647.8185","51,538.0433","66,690.2016","422,807.1455",813
annual turbine inspection,"411,615.7768","52,385.0533","63,925.7838","360,880.2662",830
cable major repair,"18,570.8695","11,366.4014",0.0000,"7,285.5439",21
cable replacement,"24,841.9109","17,360.8987",0.0000,"7,534.1609",20
inspection,"31,894.2584",316.3979,0.0000,"31,610.7584",10
main shaft major repair,"1,896.0713","1,685.8569","1,705.2425",268.3213,17
main shaft minor repair,"6,586.3096","3,665.5071","4,449.7793","3,232.5596",168
main shaft replacement,"8,534.9010","1,620.5272","8,534.9010","6,933.1001",6
major pitch system repair,"13,655.3076","11,950.8297","12,068.3193","1,986.0576",123


  Scheduled Task Completion Rate: 99.61%
Unscheduled Task Completion Rate: 99.44%
    Overall Task Completion Rate: 99.50%
{'name': 'fixed_bottom_15_MW_-5%_fixed_bottom', 'library': 'library', 'weather': 'boem_era5_41.0_-70.5.csv', 'service_equipment': ['ctv1.yaml', 'ctv2.yaml', 'ctv3.yaml', 'clv.yaml', 'jackup.yaml', 'dsv.yaml'], 'layout': 'fixed_bottom_layout_15_MW_-5%.csv', 'inflation_rate': 0, 'fixed_costs': 'fixed_bottom_fixed_costs.yaml', 'workday_start': 7, 'workday_end': 19, 'start_year': 2000, 'end_year': 2020, 'project_capacity': 600, 'port': 'port116.yaml', 'random_generator': Generator(PCG64) at 0x275A9E2CAC0}


...

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,time_to_completion,process_time,downtime,time_to_start,N
category,,,,,
annual inspection,"463,172.0949","51,456.4664","82,751.1677","413,389.9641",806
annual turbine inspection,"414,820.6854","53,203.2787","80,131.6395","363,353.3574",843
cable major repair,"18,608.2945","11,122.7040",0.0000,"6,613.5558",20
cable replacement,"21,044.2736","14,502.8589",0.0000,"6,582.2736",17
inspection,"33,910.4301",305.7943,0.0000,"33,626.9301",10
main shaft major repair,"2,503.3176","2,250.0131","2,233.2103",361.3176,23
main shaft minor repair,"8,424.4475","3,762.6854","5,731.0012","4,989.6592",182
main shaft replacement,"4,113.9780","1,065.6028","4,113.9780","3,059.4780",4
major pitch system repair,"19,013.7282","15,979.0826","16,883.5278","3,222.7279",159


  Scheduled Task Completion Rate: 99.56%
Unscheduled Task Completion Rate: 99.43%
    Overall Task Completion Rate: 99.47%


,fixed_bottom_15_MW_-10%,fixed_bottom_15_MW_-5%,fixed_bottom_15_MW_baseline
Metrics,,,
# Turbines,40.0000,40.0000,40.0000
Turbine Rating (MW),15.0000,15.0000,15.0000
Project Capacity (MW),600.0000,600.0000,600.0000
# OSS,1.0000,1.0000,1.0000
Total Export Cable Length (km),118.0680,118.0680,118.0680
Total Array Cable Length (km),196.5491,196.5491,196.5491
FCR (%),0.0648,0.0648,0.0648
CapEx ($),"2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287"
CapEx per kW ($/kW),"3,837.1480","3,837.1480","3,837.1480"


time taken to run: 1041.2302518999786


In [5]:
display(summary_results)

,fixed_bottom_15_MW_-10%,fixed_bottom_15_MW_-5%,fixed_bottom_15_MW_baseline
Metrics,,,
# Turbines,40.0000,40.0000,40.0000
Turbine Rating (MW),15.0000,15.0000,15.0000
Project Capacity (MW),600.0000,600.0000,600.0000
# OSS,1.0000,1.0000,1.0000
Total Export Cable Length (km),118.0680,118.0680,118.0680
Total Array Cable Length (km),196.5491,196.5491,196.5491
FCR (%),0.0648,0.0648,0.0648
CapEx ($),"2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287"
CapEx per kW ($/kW),"3,837.1480","3,837.1480","3,837.1480"


In [6]:
summary_results.to_csv('library/results/task_completion_15_MW_LCOE_Breakdown.csv')

In [7]:
def display_results_all_runs_2(list_projects):
    import pandas as pd
    library_path = Path("library").resolve()
    df_list_fixed_bottom = list()
    df_list_floating = list()
    df_final_list = list()
    num = 0
    for i in list_projects:
        config = load_yaml(
            library_path / "project/config",
            f"{i.replace(' ', '_')}_base.yaml"
        )

        wombat_config = load_yaml(
            library_path / "project/config",
            f"{i.replace(' ', '_')}_base_operations.yaml"
        )
    
    
        wombat_config['random_generator'] = np.random.default_rng(seed=34)
    
        print(wombat_config)
        config["wombat_config"] = wombat_config
        for j in range(0,2):
            
            df_2 = extract_results(i, config)
            
            if num == 0:
                df_list_floating = df_2
            else:
                df_list_floating = pd.concat([df_list_floating, df_2], axis=1)
                display(df_list_floating)
            
            num = num + 1        

            
    final_df = df_list_floating
                          
    return final_df

In [8]:
import time
t1 = time.perf_counter()

list_projects_15_MW = ["fixed_bottom_15_MW_-10%", "fixed_bottom_15_MW_-5%", "fixed_bottom_15_MW_baseline"]
summary_results = display_results_all_runs_2(list_projects_15_MW)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

{'name': 'fixed_bottom_15_MW_-10%_fixed_bottom', 'library': 'library', 'weather': 'boem_era5_41.0_-70.5.csv', 'service_equipment': ['ctv1.yaml', 'ctv2.yaml', 'ctv3.yaml', 'clv.yaml', 'jackup.yaml', 'dsv.yaml'], 'layout': 'fixed_bottom_layout_15_MW_-10%.csv', 'inflation_rate': 0, 'fixed_costs': 'fixed_bottom_fixed_costs.yaml', 'workday_start': 7, 'workday_end': 19, 'start_year': 2000, 'end_year': 2020, 'project_capacity': 600, 'port': 'port116.yaml', 'random_generator': Generator(PCG64) at 0x275A1FDA260}


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\upsizing\ORBIT\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,time_to_completion,process_time,downtime,time_to_start,N
category,,,,,
annual inspection,"472,647.8185","51,538.0433","66,690.2016","422,807.1455",813
annual turbine inspection,"411,615.7768","52,385.0533","63,925.7838","360,880.2662",830
cable major repair,"18,570.8695","11,366.4014",0.0000,"7,285.5439",21
cable replacement,"24,841.9109","17,360.8987",0.0000,"7,534.1609",20
inspection,"31,894.2584",316.3979,0.0000,"31,610.7584",10
main shaft major repair,"1,896.0713","1,685.8569","1,705.2425",268.3213,17
main shaft minor repair,"6,586.3096","3,665.5071","4,449.7793","3,232.5596",168
main shaft replacement,"8,534.9010","1,620.5272","8,534.9010","6,933.1001",6
major pitch system repair,"13,655.3076","11,950.8297","12,068.3193","1,986.0576",123


  Scheduled Task Completion Rate: 99.61%
Unscheduled Task Completion Rate: 99.44%
    Overall Task Completion Rate: 99.50%


...

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,time_to_completion,process_time,downtime,time_to_start,N
category,,,,,
annual inspection,"477,539.4350","51,539.3827","68,442.6385","427,856.9350",808
annual turbine inspection,"404,875.4818","55,286.6992","65,811.4827","350,693.1911",859
cable major repair,"23,147.4897","14,074.2039",0.0000,"9,152.4897",24
cable replacement,"23,031.7527","16,554.9846",0.0000,"6,541.0027",19
inspection,"33,533.0419",282.7975,0.0000,"33,277.5419",10
main shaft major repair,"2,646.4878","2,334.8568","2,406.3297",399.2378,24
main shaft minor repair,"7,034.8251","4,240.0647","4,449.9714","3,179.3251",190
main shaft replacement,"7,442.3867","2,656.6133","7,442.3867","5,076.6367",10
major pitch system repair,"15,152.8080","13,467.0530","13,648.4038","1,790.3080",140


  Scheduled Task Completion Rate: 99.81%
Unscheduled Task Completion Rate: 99.36%
    Overall Task Completion Rate: 99.50%


,fixed_bottom_15_MW_-10%,fixed_bottom_15_MW_-10%,fixed_bottom_15_MW_-5%,fixed_bottom_15_MW_-5%,fixed_bottom_15_MW_baseline,fixed_bottom_15_MW_baseline
Metrics,,,,,,
# Turbines,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000
Turbine Rating (MW),15.0000,15.0000,15.0000,15.0000,15.0000,15.0000
Project Capacity (MW),600.0000,600.0000,600.0000,600.0000,600.0000,600.0000
# OSS,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Total Export Cable Length (km),118.0680,118.0680,118.0680,118.0680,118.0680,118.0680
Total Array Cable Length (km),196.5491,196.5491,196.5491,196.5491,196.5491,196.5491
FCR (%),0.0648,0.0648,0.0648,0.0648,0.0648,0.0648
CapEx ($),"2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287","2,302,288,824.0287"
CapEx per kW ($/kW),"3,837.1480","3,837.1480","3,837.1480","3,837.1480","3,837.1480","3,837.1480"


time taken to run: 2236.8749993999954
